# Notebook 09 - Prosodic Agent Prototype

## Goal
Build a prosodic baseline with interpretable feature vectors.


## Agenda
- Extract F0 and energy stats
- Build simple table
- Train interpretable classifier
- Inspect behavior


## Concept and Math

Prosodic cues capture source and timing behavior that may remain unstable in synthesis.
Linear baselines let you reason about feature influence directly.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa as lb
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_ROOT = Path("../dataset")
audio_files = sorted(DATA_ROOT.rglob("*.flac")) + sorted(DATA_ROOT.rglob("*.wav"))

rows = []
for p in audio_files:
    s = str(p).lower()
    if "fake" in s or "spoof" in s:
        label = 1
    elif "real" in s or "bona" in s or "human" in s:
        label = 0
    else:
        continue
    w, sr = lb.load(p, sr=16000, mono=True)
    f0, _, _ = lb.pyin(w, fmin=lb.note_to_hz("C2"), fmax=lb.note_to_hz("C7"), sr=sr, hop_length=160)
    f0v = f0[~np.isnan(f0)]
    rms = lb.feature.rms(y=w, hop_length=160)[0]
    rows.append({
        "f0_mean": np.nanmean(f0v) if len(f0v) else np.nan,
        "f0_std": np.nanstd(f0v) if len(f0v) else np.nan,
        "rms_mean": float(np.mean(rms)),
        "rms_std": float(np.std(rms)),
        "label": label,
    })

df = pd.DataFrame(rows).dropna()
print("rows:", len(df))
if len(df) >= 20 and df["label"].nunique() == 2:
    X = df[["f0_mean", "f0_std", "rms_mean", "rms_std"]].values
    y = df["label"].values
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(X, y)
    print("train_accuracy:", model.score(X, y))


## PyTorch Equivalent Snippet
Understand the librosa block first, then map it to this snippet.


In [ ]:
import torch
import torch.nn as nn

linear = nn.Linear(4, 2)
print(linear(torch.randn(5, 4)).shape)


## Review Checklist
- Which prosodic features are most stable?
- Why can high train accuracy be misleading?
- How would you interpret coefficients in a linear model?
